# Notebook 02 — SSE por Baixo dos Panos

O Stage 02 do curso usa `sse-starlette` para transmitir tokens do agente ao cliente.
Mas o que o FastAPI realmente envia pela rede?

Este notebook responde isso construindo SSE do zero, sem frameworks,
para que voce entenda o protocolo antes de ver a abstracao.

## O que e SSE?

**Server-Sent Events (SSE)** e um protocolo padronizado pelo W3C que permite ao servidor
enviar mensagens ao cliente de forma unidirecional, sobre uma conexao HTTP comum.

Caracteristicas principais:

- **Unidirecional** — somente o servidor envia dados; o cliente nao envia mensagens pela mesma conexao.
- **Sobre HTTP/1.1 ou HTTP/2** — nao exige upgrade de protocolo (ao contrario de WebSocket).
- **Reconexao automatica** — o browser tenta reconectar automaticamente se a conexao cair, usando o campo `id` para retomar de onde parou.
- **Texto puro** — os eventos sao linhas de texto UTF-8, faceis de inspecionar com `curl`.
- **Suporte nativo no browser** — a API `EventSource` esta disponivel em todos os browsers modernos sem bibliotecas extras.

## Anatomia de um evento SSE

Um evento SSE e um bloco de linhas de texto terminado por uma linha em branco.
Cada linha tem o formato `campo: valor`.

```
id: 42
event: token
data: {"content": "Ola, mundo!"}

```

Campos disponiveis:

| Campo | Obrigatorio | Descricao |
|---|---|---|
| `data` | Sim | Conteudo do evento. Pode ter multiplas linhas `data:`. |
| `event` | Nao | Tipo do evento. Padrao: `message`. |
| `id` | Nao | Identificador do evento. O browser envia de volta no header `Last-Event-ID` na reconexao. |
| `retry` | Nao | Tempo em ms para o browser esperar antes de reconectar. |

Linhas que comecam com `:` sao comentarios — o servidor as usa para manter a conexao viva
(keep-alive) sem emitir um evento real:

```
: keep-alive

```

In [ ]:
# Construindo o formato SSE manualmente em Python
# Isso e exatamente o que o servidor envia pela rede, byte a byte.

def formatar_evento_sse(
    data: str,
    event: str | None = None,
    event_id: str | None = None,
    retry: int | None = None,
) -> str:
    """Formata um evento SSE segundo a especificacao W3C."""
    linhas = []

    if event_id is not None:
        linhas.append(f"id: {event_id}")

    if event is not None:
        linhas.append(f"event: {event}")

    if retry is not None:
        linhas.append(f"retry: {retry}")

    # Dados podem ter multiplas linhas — cada uma deve ser prefixada com 'data:'
    for linha in data.splitlines():
        linhas.append(f"data: {linha}")

    # Linha em branco obrigatoria ao final — sinaliza fim do evento
    linhas.append("")
    linhas.append("")

    return "\n".join(linhas)


# Exemplo 1: evento simples de token
import json

evento_token = formatar_evento_sse(
    data=json.dumps({"content": "Ola, mundo!"}),
    event="token",
    event_id="1",
)
print("=== Evento token ===")
print(repr(evento_token))  # Mostra \n de forma explicita
print()
print(evento_token)         # Mostra como aparece no terminal

# Exemplo 2: evento de conclusao
evento_done = formatar_evento_sse(
    data=json.dumps({"tools_used": ["search_database"], "duration_ms": 1240}),
    event="done",
    event_id="99",
)
print("=== Evento done ===")
print(evento_done)

# Exemplo 3: comentario keep-alive
keep_alive = ": keep-alive\n\n"
print("=== Keep-alive ===")
print(repr(keep_alive))

## SSE vs WebSocket

A pergunta mais comum ao aprender SSE e: "por que nao usar WebSocket?".
A resposta depende do que voce precisa.

| Caracteristica | SSE | WebSocket |
|---|---|---|
| Direcao da comunicacao | Somente servidor → cliente | Bidirecional |
| Protocolo base | HTTP padrao | Upgrade para ws:// / wss:// |
| Reconexao automatica | Sim (nativa no browser) | Nao (precisa implementar) |
| Suporte a proxies/CDNs | Excelente (e HTTP) | Variavel (alguns proxies nao suportam) |
| Multiplexacao (HTTP/2) | Sim | Nao |
| Complexidade de implementacao | Baixa | Maior |
| Caso de uso ideal | Notificacoes, streaming de respostas de LLM | Chat bidirecional, jogos, colaboracao em tempo real |

Para o caso de uso do curso — transmitir tokens de um agente ao cliente — SSE e a escolha certa:
o fluxo e estritamente unidirecional e se beneficia da simplicidade e do suporte universal a HTTP.

In [ ]:
# Producer SSE: gerador assincrono que produz eventos formatados
# Este e o padrao exato usado no main.py do Stage 02

import asyncio

async def sse_producer(tokens: list[str]):
    """Gera eventos SSE formatados a partir de uma lista de tokens."""
    for i, token in enumerate(tokens):
        payload = json.dumps({"content": token})
        yield formatar_evento_sse(data=payload, event="token", event_id=str(i))
        # Simula latencia de rede / geracao do modelo
        await asyncio.sleep(0.05)

    # Evento de conclusao
    done_payload = json.dumps({"status": "ok", "total_tokens": len(tokens)})
    yield formatar_evento_sse(data=done_payload, event="done", event_id=str(len(tokens)))


# Simula o que o servidor enviaria para um cliente real
tokens_simulados = ["Ola", ", ", "eu", " sou", " o", " agente", "."]

print("Bytes que o servidor envia pela rede:\n")
print("-" * 50)

async def demo_producer():
    async for evento in sse_producer(tokens_simulados):
        print(evento, end="")

await demo_producer()

In [ ]:
# Consumer SSE: parse de eventos a partir de uma stream de bytes HTTP
# Mostra como um cliente Python leria e interpretaria os eventos

def parse_sse_chunk(raw: str) -> dict | None:
    """
    Faz parse de um bloco SSE (tudo entre duas linhas em branco).
    Retorna um dict com os campos encontrados, ou None para comentarios.
    """
    fields: dict[str, list[str]] = {}

    for linha in raw.strip().splitlines():
        if linha.startswith(":"):  # Comentario — ignora
            return None

        if ":" in linha:
            campo, _, valor = linha.partition(":")
            campo = campo.strip()
            valor = valor.lstrip(" ")  # Um espaco opcional apos os dois-pontos
            fields.setdefault(campo, []).append(valor)

    if not fields:
        return None

    # Junta multiplas linhas 'data' com newline
    resultado = {campo: "\n".join(valores) for campo, valores in fields.items()}
    return resultado


def consumer_sse(stream_bruta: str):
    """
    Processa uma stream SSE completa (varios eventos concatenados).
    Divide pelos separadores de evento (linha dupla em branco) e faz parse de cada um.
    """
    # Divide a stream em blocos de eventos
    blocos = stream_bruta.split("\n\n")

    for bloco in blocos:
        bloco = bloco.strip()
        if not bloco:
            continue
        evento = parse_sse_chunk(bloco)
        if evento:
            yield evento


# Monta uma stream simulada com os mesmos eventos do producer acima
stream_simulada = ""
for token in tokens_simulados:
    payload = json.dumps({"content": token})
    stream_simulada += formatar_evento_sse(data=payload, event="token")

stream_simulada += formatar_evento_sse(
    data=json.dumps({"status": "ok"}),
    event="done",
)

# Processa e exibe
print("Eventos parseados pelo consumer:\n")
texto_completo = ""
for evento in consumer_sse(stream_simulada):
    tipo = evento.get("event", "message")
    dados = json.loads(evento.get("data", "{}"))

    if tipo == "token":
        texto_completo += dados.get("content", "")
        print(f"  token -> {dados['content']!r}")
    elif tipo == "done":
        print(f"  done  -> {dados}")

print(f"\nTexto reconstruido: {texto_completo!r}")

## Como o FastAPI abstrai isso

O `main.py` do Stage 02 usa `sse-starlette`, que esconde toda a formatacao manual vista acima.
Voce so precisa:

1. Criar um gerador assincrono que yielda `ServerSentEvent` (ou dicts)
2. Retornar um `EventSourceResponse` na rota

A biblioteca cuida de: cabecalhos HTTP corretos, encoding UTF-8, linha em branco entre eventos
e keep-alive automatico.

In [ ]:
# Comparacao lado a lado: abordagem manual vs abstracao do FastAPI

print("=" * 60)
print("ABORDAGEM MANUAL (o que vimos neste notebook)")
print("=" * 60)

codigo_manual = '''
# Formata manualmente
async def gerar_eventos(tokens):
    for i, token in enumerate(tokens):
        payload = json.dumps({"content": token})
        evento = f"event: token\\n"
        evento += f"id: {i}\\n"
        evento += f"data: {payload}\\n\\n"
        yield evento.encode("utf-8")

# Retorna StreamingResponse com content-type correto
@app.get("/stream")
async def endpoint_manual(pergunta: str):
    tokens = obter_tokens(pergunta)
    return StreamingResponse(
        gerar_eventos(tokens),
        media_type="text/event-stream",
        headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"},
    )
'''
print(codigo_manual)

print("=" * 60)
print("ABSTRACAO DO FASTAPI com sse-starlette (Stage 02)")
print("=" * 60)

codigo_fastapi = '''
from sse_starlette.sse import EventSourceResponse, ServerSentEvent

# Gerador retorna objetos ServerSentEvent
async def gerar_eventos(tokens):
    for i, token in enumerate(tokens):
        yield ServerSentEvent(
            data=json.dumps({"content": token}),
            event="token",
            id=str(i),
        )

# EventSourceResponse cuida de headers, encoding e keep-alive
@app.get("/stream")
async def endpoint_fastapi(pergunta: str):
    tokens = obter_tokens(pergunta)
    return EventSourceResponse(gerar_eventos(tokens))
'''
print(codigo_fastapi)

print()
print("Resultado na rede: identico em ambos os casos.")
print("A abstracao so elimina o trabalho manual de formatacao.")

## EventSource no browser

No lado do cliente, o browser expoe a API `EventSource` que faz a conexao e dispatch
dos eventos automaticamente. Nao e necessario nenhuma biblioteca.

```javascript
// Abre a conexao SSE
const source = new EventSource("/stream?pergunta=Qual+o+total+de+vendas");

// Ouve eventos do tipo 'token'
source.addEventListener("token", (event) => {
    const dados = JSON.parse(event.data);
    document.getElementById("resposta").textContent += dados.content;
});

// Ouve o evento de conclusao
source.addEventListener("done", (event) => {
    const dados = JSON.parse(event.data);
    console.log("Concluido:", dados);
    source.close();  // Fecha a conexao apos receber o done
});

// Ouve erros (reconexao e automatica)
source.onerror = (err) => {
    console.error("Erro SSE:", err);
};
```

O browser reenvia automaticamente o header `Last-Event-ID` na reconexao,
permitindo que o servidor retome o stream de onde parou (se implementado).

## Tente voce

Tres exercicios para consolidar o aprendizado:

**Exercicio 1 — Multiplas linhas de data:**
Modifique `formatar_evento_sse()` para aceitar `data` como uma string com quebras de linha.
Verifique que cada linha e prefixada com `data:` separadamente.
Adapte `parse_sse_chunk()` para juntar as linhas corretamente.

**Exercicio 2 — Retry automatico:**
Adicione um campo `retry: 3000` no primeiro evento do producer.
Verifique no consumer que o campo e lido corretamente.
Pesquise: qual o valor padrao de retry nos browsers modernos quando o campo nao esta presente?

**Exercicio 3 — Keep-alive:**
Modifique o `sse_producer` para emitir um comentario `: keep-alive` a cada 2 eventos reais.
Verifique que o consumer ignora os comentarios corretamente.